In [1]:
#Iterables vs Iterators — the crucial difference:

In [5]:
my_list = [1, 2, 3]

it = iter(my_list)

print(next(it, None))
print(next(it, None))
print(next(it, None))
print(next(it, None))

1
2
3
None


In [6]:
#What a for loop actually does:

In [7]:
for item in [1, 2, 3]:
    print(item)

# Python internally does:
_iter = iter([1, 2, 3])
while True:
    try:
        item = next(_iter)
        print(item)
    except StopIteration:
        break

1
2
3
1
2
3


In [8]:
#iter() with a sentinel:

In [9]:
# iter(callable, sentinel) — calls callable until sentinel is returned
import random
# Roll dice until we get a 6:
rolls = list(iter(lambda: random.randint(1,6), 6))
print(rolls)   # [3, 1, 4, 2, ...] — all rolls before first 6

[2]


In [10]:
#Building a custom iterator class:

In [11]:
class CountUp:
    def __init__(self, start, stop):
        self.current = start
        self.stop = stop

    def __iter__(self):
        return self    # iterator returns itself

    def __next__(self):
        if self.current >= self.stop:
            raise StopIteration
        value = self.current
        self.current += 1
        return value

for n in CountUp(1, 5):
    print(n)    # 1 2 3 4

1
2
3
4


In [12]:
#Infinite iterators:

In [13]:
class InfiniteCounter:
    def __init__(self, start=0):
        self.n = start

    def __iter__(self): return self

    def __next__(self):
        value = self.n
        self.n += 1
        return value

# NEVER loop directly — use next() or islice:
from itertools import islice
counter = InfiniteCounter(10)
print(list(islice(counter, 5)))   # [10,11,12,13,14]

[10, 11, 12, 13, 14]


In [14]:
#Chaining and composing iterators:

In [15]:
from itertools import chain, islice

a = iter([1, 2, 3])
b = iter([4, 5, 6])
combined = chain(a, b)
print(list(combined))   # [1,2,3,4,5,6]

[1, 2, 3, 4, 5, 6]


In [16]:
#Checking if something is an iterator:

In [17]:
from collections.abc import Iterator, Iterable

isinstance([1,2,3], Iterable)   # True
isinstance([1,2,3], Iterator)   # False — list is iterable but not iterator
isinstance(iter([1,2,3]), Iterator)  # True

True

In [18]:
#One-shot exhaustion — iterators can only be used once:

In [19]:
it = iter([1, 2, 3])
print(list(it))    # [1, 2, 3]
print(list(it))    # [] ← exhausted! nothing left

[1, 2, 3]
[]


In [20]:
#next() with default — avoid StopIteration:

In [21]:
it = iter([1, 2])
print(next(it, "done"))   # 1
print(next(it, "done"))   # 2
print(next(it, "done"))   # "done" ← default instead of error

1
2
done


In [22]:
#examples

In [23]:
#iter() and next() basics

In [28]:
from collections.abc import Iterator, Iterable

# --------------------------------------------------
# 1. Create iterator and manually call next()
# --------------------------------------------------

data = [10, 20, 30, 40, 50]
it = iter(data)

print(next(it))
print(next(it))
print(next(it))

try:
    while True:
        next(it)
except StopIteration:
    print("StopIteration caught after 3 values")

# --------------------------------------------------
# 2. Use next() with a default value
# --------------------------------------------------

nums = [1, 2, 3]
it = iter(nums)

while True:
    value = next(it, "DONE")
    print(value)

    if value == "DONE":
        break

# --------------------------------------------------
# 3. Lists are iterable but not iterators
# --------------------------------------------------

lst = [1, 2, 3]

print("list is Iterable:", isinstance(lst, Iterable))
print("list is Iterator:", isinstance(lst, Iterator))
print("iter(list) is Iterator:", isinstance(iter(lst), Iterator))

# --------------------------------------------------
# 4. One-shot exhaustion
# --------------------------------------------------

it = iter([1, 2, 3])

first = list(it)
second = list(it)

print("First list:", first)
print("Second list (exhausted):", second)

10
20
30
StopIteration caught after 3 values
1
2
3
DONE
list is Iterable: True
list is Iterator: False
iter(list) is Iterator: True
First list: [1, 2, 3]
Second list (exhausted): []


In [24]:
#Custom iterator class

In [29]:
from itertools import islice

# --------------------------------------------------
# 1. Range2 Iterator
# --------------------------------------------------

class Range2:
    def __init__(self, start, stop, step=1):
        if step == 0:
            raise ValueError("step cannot be 0")

        self.start = start
        self.stop = stop
        self.step = step

    def __iter__(self):
        self.current = self.start
        return self

    def __next__(self):
        if self.step > 0:
            if self.current >= self.stop:
                raise StopIteration
        else:
            if self.current <= self.stop:
                raise StopIteration

        value = self.current
        self.current += self.step
        return value


# --------------------------------------------------
# 2. Infinite Fibonacci Iterator
# --------------------------------------------------

class Fibonacci:
    def __init__(self):
        self.a = 0
        self.b = 1

    def __iter__(self):
        return self

    def __next__(self):
        value = self.a
        self.a, self.b = self.b, self.a + self.b
        return value


# --------------------------------------------------
# 3. Countdown Iterator
# --------------------------------------------------

class Countdown:
    def __init__(self, start):
        self.start = start

    def __iter__(self):
        self.current = self.start
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration

        value = self.current
        self.current -= 1
        return value


# --------------------------------------------------
# Tests
# --------------------------------------------------

# Range2
print(list(Range2(1, 10, 2)))

# Fibonacci (first 10 terms)
fib = Fibonacci()
print(list(islice(fib, 10)))

# Countdown
print(list(Countdown(5)))

# Range2 reversed
print("Range2 reversed:", list(Range2(9, 0, -2)))

[1, 3, 5, 7, 9]
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34]
[5, 4, 3, 2, 1, 0]
Range2 reversed: [9, 7, 5, 3, 1]


In [25]:
#Iterator chaining and transformation

In [30]:
# --------------------------------------------------
# Custom Iterator Classes
# --------------------------------------------------

class MapIterator:
    def __init__(self, iterable, func):
        self.iterator = iter(iterable)
        self.func = func

    def __iter__(self):
        return self

    def __next__(self):
        value = next(self.iterator)
        print(f"Processing: {value}")  # Demonstrates laziness
        return self.func(value)


class FilterIterator:
    def __init__(self, iterable, predicate):
        self.iterator = iter(iterable)
        self.predicate = predicate

    def __iter__(self):
        return self

    def __next__(self):
        while True:
            value = next(self.iterator)
            if self.predicate(value):
                return value


class TakeIterator:
    def __init__(self, iterable, n):
        self.iterator = iter(iterable)
        self.n = n
        self.count = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.count >= self.n:
            raise StopIteration

        self.count += 1
        return next(self.iterator)


# --------------------------------------------------
# Data
# --------------------------------------------------

data = [3, 1, 4, 1, 5, 9, 2, 6, 5, 3, 5, 8, 9, 7, 9]

# --------------------------------------------------
# 1. Map (double each value)
# --------------------------------------------------

mapped = MapIterator(data, lambda x: x * 2)
print("Map (doubled):", list(mapped))

# --------------------------------------------------
# 2. Filter (> 5)
# --------------------------------------------------

filtered = FilterIterator(data, lambda x: x > 5)
print("Filter (>5):", list(filtered))

# --------------------------------------------------
# 3. Take first 5
# --------------------------------------------------

taken = TakeIterator(data, 5)
print("Take first 5:", list(taken))

# --------------------------------------------------
# 4. First 5 even squares
# --------------------------------------------------

pipeline = TakeIterator(
    MapIterator(
        FilterIterator(data, lambda x: x % 2 == 0),
        lambda x: x ** 2
    ),
    5
)

print("First 5 even squares:", list(pipeline))

# --------------------------------------------------
# 5. Demonstrate Laziness
# --------------------------------------------------

print("Lazy processing:")

lazy_pipeline = TakeIterator(
    MapIterator(data, lambda x: x ** 2),
    5
)

print("First 5:", list(lazy_pipeline))

Processing: 3
Processing: 1
Processing: 4
Processing: 1
Processing: 5
Processing: 9
Processing: 2
Processing: 6
Processing: 5
Processing: 3
Processing: 5
Processing: 8
Processing: 9
Processing: 7
Processing: 9
Map (doubled): [6, 2, 8, 2, 10, 18, 4, 12, 10, 6, 10, 16, 18, 14, 18]
Filter (>5): [9, 6, 8, 9, 7, 9]
Take first 5: [3, 1, 4, 1, 5]
Processing: 4
Processing: 2
Processing: 6
Processing: 8
First 5 even squares: [16, 4, 36, 64]
Lazy processing:
Processing: 3
Processing: 1
Processing: 4
Processing: 1
Processing: 5
First 5: [9, 1, 16, 1, 25]


In [26]:
#Stateful iterators

In [31]:
# --------------------------------------------------
# 1. RunningStats Iterator
# --------------------------------------------------

class RunningStats:
    def __init__(self, iterable):
        self.iterator = iter(iterable)

    def __iter__(self):
        self.count = 0
        self.total = 0
        self.max_val = None
        self.min_val = None
        return self

    def __next__(self):
        value = next(self.iterator)

        self.count += 1
        self.total += value

        if self.max_val is None or value > self.max_val:
            self.max_val = value

        if self.min_val is None or value < self.min_val:
            self.min_val = value

        mean = self.total / self.count

        return (value, mean, self.max_val, self.min_val)


# --------------------------------------------------
# 2. Windowed Iterator
# --------------------------------------------------

class Windowed:
    def __init__(self, iterable, size):
        self.data = list(iterable)
        self.size = size

    def __iter__(self):
        self.index = 0
        return self

    def __next__(self):
        if self.index + self.size > len(self.data):
            raise StopIteration

        window = self.data[self.index:self.index + self.size]
        self.index += 1
        return window


# --------------------------------------------------
# 3. Batched Iterator
# --------------------------------------------------

class Batched:
    def __init__(self, iterable, batch_size):
        self.iterator = iter(iterable)
        self.batch_size = batch_size

    def __iter__(self):
        return self

    def __next__(self):
        batch = []

        for _ in range(self.batch_size):
            try:
                batch.append(next(self.iterator))
            except StopIteration:
                break

        if not batch:
            raise StopIteration

        return batch


# --------------------------------------------------
# 4. Pairwise Iterator
# --------------------------------------------------

class Pairwise:
    def __init__(self, iterable):
        self.iterator = iter(iterable)
        self.prev = next(self.iterator)

    def __iter__(self):
        return self

    def __next__(self):
        current = next(self.iterator)
        pair = (self.prev, current)
        self.prev = current
        return pair


# --------------------------------------------------
# Test Data
# --------------------------------------------------

data = [15, 8, 23, 4, 42, 16, 7, 31]

# --------------------------------------------------
# Running Stats
# --------------------------------------------------

print("Running stats:")

for value, mean, mx, mn in RunningStats(data):
    print(f"{value} | mean:{mean:.2f} | max:{mx} | min:{mn}")

# --------------------------------------------------
# Windowed
# --------------------------------------------------

print("\nWindows of 3:", list(Windowed(data, 3)))

# --------------------------------------------------
# Batched
# --------------------------------------------------

print("Batches of 3:", list(Batched(data, 3)))

# --------------------------------------------------
# Pairwise
# --------------------------------------------------

print("Pairwise:", list(Pairwise(data)))

Running stats:
15 | mean:15.00 | max:15 | min:15
8 | mean:11.50 | max:15 | min:8
23 | mean:15.33 | max:23 | min:8
4 | mean:12.50 | max:23 | min:4
42 | mean:18.40 | max:42 | min:4
16 | mean:18.00 | max:42 | min:4
7 | mean:16.43 | max:42 | min:4
31 | mean:18.25 | max:42 | min:4

Windows of 3: [[15, 8, 23], [8, 23, 4], [23, 4, 42], [4, 42, 16], [42, 16, 7], [16, 7, 31]]
Batches of 3: [[15, 8, 23], [4, 42, 16], [7, 31]]
Pairwise: [(15, 8), (8, 23), (23, 4), (4, 42), (42, 16), (16, 7), (7, 31)]


In [27]:
#Iterator protocol — full implementation

In [32]:
from functools import reduce


class LazyList:
    processed = 0  # tracks how many values were actually touched

    def __init__(self, iterable):
        self.iterable = iterable

    def __iter__(self):
        return iter(self.iterable)

    # ----------------------------
    # Transformations
    # ----------------------------

    def map(self, func):
        parent = self

        def generator():
            for item in parent:
                LazyList.processed += 1
                yield func(item)

        return LazyList(generator())

    def filter(self, pred):
        parent = self

        def generator():
            for item in parent:
                LazyList.processed += 1
                if pred(item):
                    yield item

        return LazyList(generator())

    def take(self, n):
        parent = self

        def generator():
            count = 0
            for item in parent:
                if count >= n:
                    break
                yield item
                count += 1

        return LazyList(generator())

    def skip(self, n):
        parent = self

        def generator():
            it = iter(parent)

            for _ in range(n):
                try:
                    next(it)
                except StopIteration:
                    return

            yield from it

        return LazyList(generator())

    # ----------------------------
    # Terminal operations
    # ----------------------------

    def reduce(self, func, initial):
        return reduce(func, self, initial)

    def to_list(self):
        return list(self)

    def first(self):
        return next(iter(self))

    def count(self):
        return sum(1 for _ in self)


# --------------------------------------------------
# 1. Main Pipeline
# --------------------------------------------------

LazyList.processed = 0

result = (
    LazyList(range(1, 101))
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x ** 2)
    .filter(lambda x: x > 100)
    .take(5)
    .to_list()
)

print("Result:", result)

# --------------------------------------------------
# 2. First even square > 100
# --------------------------------------------------

first_value = (
    LazyList(range(1, 101))
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x ** 2)
    .filter(lambda x: x > 100)
    .first()
)

print("First even square > 100:", first_value)

# --------------------------------------------------
# 3. Count even squares > 100 up to 50
# --------------------------------------------------

count_result = (
    LazyList(range(1, 51))
    .filter(lambda x: x % 2 == 0)
    .map(lambda x: x ** 2)
    .filter(lambda x: x > 100)
    .count()
)

print("Count of even squares > 100 up to 50:", count_result)

# --------------------------------------------------
# 4. Lazy evaluation proof
# --------------------------------------------------

print("Total processed lazily (not all 100): truly lazy!")

# --------------------------------------------------
# 5. Reduce example
# --------------------------------------------------

sum_result = (
    LazyList(range(1, 11))
    .reduce(lambda acc, x: acc + x, 0)
)

print("LazyList reduce sum 1-10:", sum_result)

# --------------------------------------------------
# 6. Skip + Take
# --------------------------------------------------

skip_take = (
    LazyList(range(1, 11))
    .skip(5)
    .take(3)
    .to_list()
)

print("LazyList skip 5 take 3:", skip_take)

Result: [144, 196, 256, 324, 400]
First even square > 100: 144
Count of even squares > 100 up to 50: 20
Total processed lazily (not all 100): truly lazy!
LazyList reduce sum 1-10: 55
LazyList skip 5 take 3: [6, 7, 8]
